In [12]:
# Fear Conditioningのデータを読み込み

from pathlib import Path  # noqa: F811

import numpy as np  # noqa: F811
import pandas as pd

# 自動でエクセルファイルを読み込む

folder = r"C:\Users\sryoh\Documents\Python_MSSE_analysis\FC"

files = sorted(Path(folder).glob("*.xls")) # フォルダ内の .xls ファイルを取得してソート

if not files:
    print(f"{folder} の中に .xls ファイルはありません")
else:
    dfs = [] # 空のリストを作成して、各ファイルのデータフレームを格納
    for file in files: # ファイルごとにループ
        df = (
        pd.read_excel(file) 
        .iloc[2:8, [4]]  # 2行目から7行目まで、4列目を抽出
        .assign(
            No = lambda df: Path(file).stem, # No列にpathからファイル名を抽出して追加
            Group = lambda df: np.select( # Group列に条件に応じた値を追加
                condlist=[ # 条件のリスト：No列に各文字列が含まれているか
                df['No'].str.contains('SED'),
                df['No'].str.contains('LIE'),
                df['No'].str.contains('MOE')
            ],
            choicelist=['SED', 'LIE', 'MOE'], # 条件にマッチしたときに入れる値のリスト
            default='Other' # どれにも当てはまらない場合のデフォルト値
            ),
            Time  = lambda df: list(range(1, len(df) + 1)), # Time列に1から行数までの連番を追加
        )
        .assign(Freezing = lambda df: df['Interval.3'] / 60 * 100 ) # Freezing Time (%) を計算し列に追加
        .iloc[:, 1:5]
        )
        dfs.append(df) # データフレームをリストに追加

    dataFC = pd.concat(dfs, ignore_index=True) # リスト内のデータフレームを縦に結合して1つのデータフレームにする

    print(f"{len(files)} 件の .xls ファイルを読み込みました") # 読み込んだファイル数を表示
    print(dataFC) # データフレームの内容を表示

24 件の .xls ファイルを読み込みました
           No Group  Time   Freezing
0    100FCMOE   MOE     1        0.0
1    100FCMOE   MOE     2        0.0
2    100FCMOE   MOE     3        0.0
3    100FCMOE   MOE     4       47.6
4    100FCMOE   MOE     5       43.5
..        ...   ...   ...        ...
139   99FCMOE   MOE     2        0.0
140   99FCMOE   MOE     3        0.0
141   99FCMOE   MOE     4  14.566667
142   99FCMOE   MOE     5       60.8
143   99FCMOE   MOE     6  71.966667

[144 rows x 4 columns]


In [13]:
# Fear Extinctionのデータを読み込み

from pandas.core.arrays import categorical
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# 旧形式 .xls の OLE2 警告を抑制
# warnings.filterwarnings("ignore", message=".*OLE2 inconsistency.*")

def infer_group(name: str) -> str: # ファイル名からグループを推測する関数
    if "SED" in name:
        return "SED"
    elif "LIE" in name:
        return "LIE"
    elif "MOE" in name:
        return "MOE"
    return "Other"

def read_extinction_per3(files):   # 3分ごとのデータを読み込む関数
    rows = [] # 空のリストを作成して、各ファイルのデータを格納

    for file in files:
        # 旧 .xls では pandas で警告が出ることがあるので抑制
        # with warnings.catch_warnings():
            # warnings.filterwarnings("ignore", message=".*OLE2 inconsistency.*")
        df = pd.read_excel(file)

        col5 = pd.to_numeric(df.iloc[:, 4], errors="coerce")  # 5列目（E列）
        bins = [ # 3分毎のbinの範囲とラベル
            (3, 8, "3"),
            (9, 14, "6"),
            (15, 20, "9"),
            (21, 26, "12"),
            (27, 32, "15"),
        ]

        for start, end, time_label in bins: # 3分毎のbinごとにデータを処理
            freezing = (
                col5.iloc[start - 1:end] # 3:8, 9:14, ...
                .astype(float)           # 数値に変換
                .mul(100 / 30)           # 30秒ごとのデータを3分ごとの平均に変換
                .mean()                  # 平均値を計算  
            )
            rows.append({
                "No": Path(file).stem,   # ファイル名を追加
                "Group": infer_group(Path(file).stem), # グループを推測して追加
                "Time": time_label,      # ラベルを追加
                "Freezing": freezing,    # 平均値を追加
            })

    return pd.DataFrame(rows)

# 既存のフォルダ
folder_Ex1 = r"C:\Users\sryoh\Documents\Python_MSSE_analysis\Ex1"
folder_Ex2 = r"C:\Users\sryoh\Documents\Python_MSSE_analysis\Ex2"

files_Ex1 = sorted(Path(folder_Ex1).glob("*.xls")) # フォルダ内の .xls ファイルを取得してソート
files_Ex2 = sorted(Path(folder_Ex2).glob("*.xls")) # フォルダ内の .xls ファイルを取得してソート

# 3分ごとのデータ
if not files_Ex1 and not files_Ex2:
    print("指定されたフォルダに .xls ファイルはありません")
else:
    dataEx1_per3 = read_extinction_per3(files_Ex1) # Ex1の3分ごとのデータを読み込む
    dataEx2_per3 = read_extinction_per3(files_Ex2) # Ex2の3分ごとのデータを読み込む

    print(f"Ex1: {len(files_Ex1)} 件, Ex2: {len(files_Ex2)} 件") 
    print(dataEx1_per3.head()) 
    print(dataEx2_per3.head())

WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but 

In [20]:
# pingouinライブラリを用いて統計検定

import pingouin as pg


# -------------------------------------------------------------------------
# 2. 対応のある二次元配置分散分析の実行 (Mauchlyの検定・Huynh-Feldt補正含む)
# -------------------------------------------------------------------------
print("=== 対応のある二次元配置分散分析 (Repeated Measures ANOVA) ===")
# pingouinのrm_anovaを使用
anova_FC = pg.mixed_anova(
    data=dataFC, 
    dv='Freezing', 
    between='Group', # 水準間要因
    within='Time',   # 水準内要因
    subject='No'     # 個体識別列
    )
print(anova_FC)
# ※ 'eps' はGreenhouse-Geisserのε。pingouinは自動で球面性を判定し、
# 必要に応じてHuynh-Feldt(HF)やGreenhouse-Geisser(GG)の調整p値を出力します。


=== 対応のある二次元配置分散分析 (Repeated Measures ANOVA) ===


ValueError: DV must be numeric.

In [21]:
import pingouin as pg
import pandas as pd

# ① Freezing 列を明示的に数値型 (float) に変換
dataFC['Freezing'] = pd.to_numeric(dataFC['Freezing'], errors='coerce').astype(float)

# ② 欠損値（NaN）があれば除外（念のため）
dataFC = dataFC.dropna(subset=['Freezing'])

# ③ データ型を確認（Freezing が float64 になっていればOK）
print("--- 列のデータ型 ---")
print(dataFC.dtypes)
print("--------------------\n")

print("=== 混合二元配置分散分析 (Mixed ANOVA) ===")

# ④ 混合二元配置分散分析 (Mixed ANOVA) を実行
anova_FC = pg.mixed_anova(
    data=dataFC, 
    dv='Freezing', 
    between='Group', # 被験者間要因（群: SED, LIE, MOE）
    within='Time',   # 被験者内要因（測定時間: 1~6）
    subject='No'     # 個体識別列（被験者ID）
)

# ⑤ 結果の表示
print(anova_FC)

--- 列のデータ型 ---
No              str
Group           str
Time          int64
Freezing    float64
dtype: object
--------------------

=== 混合二元配置分散分析 (Mixed ANOVA) ===
        Source            SS  DF1  DF2            MS          F         p_unc     p_GG_corr       np2       eps sphericity  \
0        Group    113.542654    2   21     56.771327   0.149701  8.618761e-01           NaN  0.014057       NaN        NaN   
1         Time  84911.381358    5  105  16982.276272  88.603689  4.591280e-36  3.535584e-24  0.808401  0.631812      False   
2  Interaction   2943.335216   10  105    294.333522   1.535662  1.369809e-01           NaN  0.127593       NaN        NaN   

    W_spher   p_spher  
0       NaN       NaN  
1  0.157955  0.000401  
2       NaN       NaN  


In [18]:

print(dataFC.columns.tolist())
print(dataFC.reset_index(drop=True))

['No', 'Group', 'Time', 'Freezing']
           No Group  Time   Freezing
0    100FCMOE   MOE     1        0.0
1    100FCMOE   MOE     2        0.0
2    100FCMOE   MOE     3        0.0
3    100FCMOE   MOE     4       47.6
4    100FCMOE   MOE     5       43.5
..        ...   ...   ...        ...
139   99FCMOE   MOE     2        0.0
140   99FCMOE   MOE     3        0.0
141   99FCMOE   MOE     4  14.566667
142   99FCMOE   MOE     5       60.8
143   99FCMOE   MOE     6  71.966667

[144 rows x 4 columns]
